# Streaming Vega

In the example below, we load version 6 of the vega and vega-embed libraries from [esm.sh](https://esm.sh). While vega-embed is sufficient for creating static graphs, it seems the vega library is what exposes the changeset() function. 

In [0]:
// Dynamic Imports
const vega = await import('https://esm.sh/vega@6');
const { default: embed } = await import('https://esm.sh/vega-embed');

// Vega Specification
const specification = {
  $schema: 'https://vega.github.io/schema/vega-lite/v6.json',
  data: {name: 'table'},
  width: 400,
  mark: 'point',
  encoding: {
    x: {field: 'x', type: 'quantitative', scale: {zero: false}},
    y: {field: 'y', type: 'quantitative'},
    color: {field: 'category', type: 'nominal'},
  },
};

// Embed specification in output element.
embed(output, specification).then(function(res){
  // Generates a new tuple with random walk.
  function newGenerator() {
    var counter = -1;
    var previousY = [5, 5, 5, 5];
    return function () {
      counter++;
      var newVals = previousY.map(function (v, c) {
        return {
          x: counter,
          y: Math.random(),
          category: c,
        };
      });
      previousY = newVals.map(function (v) {
        return v.y;
      });
      return newVals;
    };
  }

  // Instantiates Generator
  var valueGenerator = newGenerator();
  var minimumX = -100;

  // Calls changes() and update() every 250ms
  window.setInterval(function () {
    minimumX++;
    // create a changeset with values from the generator above
    var changeSet = vega.changeset()
      .insert(valueGenerator())
      .remove(function (t) {
        return t.x < minimumX;
      });
    
    // Update the view
    res.view.change('table', changeSet).run();
  }, 100);
})